In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [2]:
#Load data
df = pd.read_csv("/content/feature_engineered_data.csv")

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce",
    format="mixed"
)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,MonthName,Day,DayOfWeek,Hour,Quarter,IsWeekend,BasketSize,OrderValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009.0,12.0,December,1.0,Tuesday,7.0,4.0,0.0,166.0,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009.0,12.0,December,1.0,Tuesday,7.0,4.0,0.0,166.0,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009.0,12.0,December,1.0,Tuesday,7.0,4.0,0.0,166.0,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009.0,12.0,December,1.0,Tuesday,7.0,4.0,0.0,166.0,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009.0,12.0,December,1.0,Tuesday,7.0,4.0,0.0,166.0,505.3


In [3]:
#Creat churn label
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

customer = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice":"nunique",
    "Revenue":"sum"
}).reset_index()

customer.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary"
]

# Business Logic
customer["Churn"] = np.where(
    (customer["Recency"] > 30) &
    (customer["Frequency"] <= 2) &
    (customer["Monetary"] < customer["Monetary"].median()),
    1,
    0
)

In [4]:
customer["Churn"].value_counts()

,count
Churn,
0,1661
1,912


In [5]:
#Feature Target
X = customer[["Recency", "Frequency", "Monetary"]]
y = customer["Churn"]

In [6]:
#Train/ Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
#random forcast model
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, min_samples_leaf=5, n_estimators=50,
                       random_state=42)

In [8]:
#prediction
y_pred = model.predict(X_test)

In [13]:
#Acccuracy
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Accuracy :", accuracy_score(y_test, y_pred))
print()

print(classification_report(y_test, y_pred))
y_prob = model.predict_proba(X_test)[:, 1]

roc_score = roc_auc_score(y_test, y_prob)

print("ROC-AUC Score:", roc_score)

Accuracy : 1.0

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       332
           1       1.00      1.00      1.00       183

    accuracy                           1.00       515
   macro avg       1.00      1.00      1.00       515
weighted avg       1.00      1.00      1.00       515

ROC-AUC Score: 1.0


In [10]:
import pickle

with open("churn_model.pkl", "wb") as f:
    pickle.dump(model, f)

customer.to_csv("customer_churn_prediction.csv", index=False)

print("✅ Churn Prediction Completed")

✅ Churn Prediction Completed


In [11]:
print(customer[["Recency", "Frequency", "Monetary", "Churn"]].corr())

            Recency  Frequency  Monetary     Churn
Recency    1.000000  -0.276851 -0.144745  0.614498
Frequency -0.276851   1.000000  0.682298 -0.274997
Monetary  -0.144745   0.682298  1.000000 -0.151059
Churn      0.614498  -0.274997 -0.151059  1.000000
